# 01 — Chest X-ray Data Preparation

**Academic prototype only.** This notebook is part of a university final project.
It is **not** a clinical diagnostic system, is **not** validated for patient care,
and must **never** be used to diagnose, screen, or triage real patients.

## Why this notebook exists

The original Coursera / NIH course validation split has **patient leakage**:
the same `PatientId` can appear in both `train-small.csv` and `valid-small.csv`.
That makes validation metrics optimistic and unreliable.

**Our approach:**
- Keep the official `test.csv` **unchanged** for final evaluation only.
- Combine only `train-small.csv` + `valid-small.csv`.
- Rebuild a **patient-level** train/validation split with `GroupShuffleSplit`
  so no patient appears in both clean splits.

Raw files under `data/raw/` and `data_dl/` are never modified.

## 0. Setup and paths

All paths are relative to the **project root**.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "xray" / "nih"
IMAGE_DIR = RAW_DIR / "images-small"
TRAIN_CSV = RAW_DIR / "train-small.csv"
VALID_CSV = RAW_DIR / "valid-small.csv"
TEST_CSV = RAW_DIR / "test.csv"

OUT_DIR = PROJECT_ROOT / "data" / "processed" / "xray"
TRAIN_CLEAN = OUT_DIR / "train_clean.csv"
VALID_CLEAN = OUT_DIR / "valid_clean.csv"

LABEL_COLS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Effusion",
    "Emphysema", "Fibrosis", "Hernia", "Infiltration", "Mass", "Nodule",
    "Pleural_Thickening", "Pneumonia", "Pneumothorax",
]

print("Project root:", PROJECT_ROOT)
print("Image dir   :", IMAGE_DIR)
print("Train CSV   :", TRAIN_CSV)
print("Valid CSV   :", VALID_CSV)
print("Test CSV    :", TEST_CSV)
print("Output dir  :", OUT_DIR)

Project root: D:\AI Engineering\medicaldiagnostic_finalproject
Image dir   : D:\AI Engineering\medicaldiagnostic_finalproject\data\raw\xray\nih\images-small
Train CSV   : D:\AI Engineering\medicaldiagnostic_finalproject\data\raw\xray\nih\train-small.csv
Valid CSV   : D:\AI Engineering\medicaldiagnostic_finalproject\data\raw\xray\nih\valid-small.csv
Test CSV    : D:\AI Engineering\medicaldiagnostic_finalproject\data\raw\xray\nih\test.csv
Output dir  : D:\AI Engineering\medicaldiagnostic_finalproject\data\processed\xray


In [2]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

print("pandas:", pd.__version__)
import sklearn
print("sklearn:", sklearn.__version__)

pandas: 3.0.5
sklearn: 1.9.0


## 1. Load and validate the three CSV files

We expect 14 disease-label columns plus `Image` and `PatientId`.
Every referenced filename must exist under `images-small/`.
Hidden macOS files (`._*`, `.DS_Store`) are ignored when scanning the folder.

In [3]:
def load_split_csv(path: Path, name: str) -> pd.DataFrame:
    if not path.is_file():
        raise FileNotFoundError(f"Missing {name}: {path}")
    df = pd.read_csv(path)
    required = ["Image", "PatientId"] + LABEL_COLS
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")
    # Keep a stable column order for later saves
    return df[required].copy()


train_raw = load_split_csv(TRAIN_CSV, "train-small.csv")
valid_raw = load_split_csv(VALID_CSV, "valid-small.csv")
test_raw = load_split_csv(TEST_CSV, "test.csv")

print("Confirmed 14 disease-label columns:")
print(LABEL_COLS)
print()
for name, df in [("train-small", train_raw), ("valid-small", valid_raw), ("test", test_raw)]:
    print(f"{name:12s}  rows={len(df):4d}  unique patients={df['PatientId'].nunique():4d}")

Confirmed 14 disease-label columns:
['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']

train-small   rows=1000  unique patients= 928
valid-small   rows= 200  unique patients= 199
test          rows= 420  unique patients= 389


In [4]:
# Real PNGs only — exclude macOS AppleDouble / Finder junk
available_images = {
    p.name
    for p in IMAGE_DIR.iterdir()
    if p.is_file()
    and p.suffix.lower() == ".png"
    and not p.name.startswith("._")
    and p.name != ".DS_Store"
}
print(f"PNG files in images-small/ (excluding ._* / .DS_Store): {len(available_images)}")


def assert_images_exist(df: pd.DataFrame, name: str) -> None:
    missing = sorted(set(df["Image"]) - available_images)
    if missing:
        preview = ", ".join(missing[:5])
        raise FileNotFoundError(
            f"{name}: {len(missing)} referenced image(s) missing. Examples: {preview}"
        )
    print(f"[ok] {name}: all {len(df)} image paths exist")


assert_images_exist(train_raw, "train-small.csv")
assert_images_exist(valid_raw, "valid-small.csv")
assert_images_exist(test_raw, "test.csv")

PNG files in images-small/ (excluding ._* / .DS_Store): 1422
[ok] train-small.csv: all 1000 image paths exist
[ok] valid-small.csv: all 200 image paths exist
[ok] test.csv: all 420 image paths exist


In [5]:
def positive_counts(df: pd.DataFrame, name: str) -> pd.DataFrame:
    counts = df[LABEL_COLS].sum().astype(int)
    out = pd.DataFrame({"positives": counts})
    out.index.name = "disease"
    print(f"\nPositive counts — {name} (n={len(df)})")
    display(out.T)
    return out


_ = positive_counts(train_raw, "train-small")
_ = positive_counts(valid_raw, "valid-small")
_ = positive_counts(test_raw, "official test")


Positive counts — train-small (n=1000)


disease,Atelectasis,Cardiomegaly,Consolidation,Edema,Effusion,Emphysema,Fibrosis,Hernia,Infiltration,Mass,Nodule,Pleural_Thickening,Pneumonia,Pneumothorax
positives,106,20,33,16,128,13,14,2,175,45,54,21,10,38



Positive counts — valid-small (n=200)


disease,Atelectasis,Cardiomegaly,Consolidation,Edema,Effusion,Emphysema,Fibrosis,Hernia,Infiltration,Mass,Nodule,Pleural_Thickening,Pneumonia,Pneumothorax
positives,19,4,7,2,27,3,1,1,29,9,11,6,1,10



Positive counts — official test (n=420)


disease,Atelectasis,Cardiomegaly,Consolidation,Edema,Effusion,Emphysema,Fibrosis,Hernia,Infiltration,Mass,Nodule,Pleural_Thickening,Pneumonia,Pneumothorax
positives,60,50,53,50,53,56,61,50,59,60,54,58,50,55


## 2. Demonstrate leakage in the original course split

Patients shared between the original train and validation CSVs are the reason
we rebuild the split.

In [6]:
orig_overlap = set(train_raw["PatientId"]) & set(valid_raw["PatientId"])
print(f"PatientId overlap between original train-small and valid-small: {len(orig_overlap)}")
if len(orig_overlap):
    print("Example leaked PatientIds:", sorted(orig_overlap)[:10])

PatientId overlap between original train-small and valid-small: 197
Example leaked PatientIds: [121, 375, 443, 589, 658, 720, 918, 1168, 1212, 1232]


## 3. Combine train-small + valid-small only

`test.csv` is **not** included here. It stays reserved for final evaluation.

In [7]:
combined = pd.concat([train_raw, valid_raw], ignore_index=True)
print(f"Combined rows (train-small + valid-small only): {len(combined)}")
print(f"Unique patients in combined pool: {combined['PatientId'].nunique()}")
print(f"Official test rows left untouched: {len(test_raw)}")

Combined rows (train-small + valid-small only): 1200
Unique patients in combined pool: 930
Official test rows left untouched: 420


## 4. Patient-level train / validation split

Use `GroupShuffleSplit` so entire patients stay in one split:
- `groups = PatientId`
- `test_size = 0.20` (this becomes the clean validation set)
- `random_state = 42`

Outputs (derived data only; raw CSVs are never overwritten):
- `data/processed/xray/train_clean.csv`
- `data/processed/xray/valid_clean.csv`

In [8]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, valid_idx = next(
    splitter.split(combined, groups=combined["PatientId"])
)

train_clean = combined.iloc[train_idx].reset_index(drop=True)
valid_clean = combined.iloc[valid_idx].reset_index(drop=True)

train_clean.to_csv(TRAIN_CLEAN, index=False)
valid_clean.to_csv(VALID_CLEAN, index=False)

print(f"Saved: {TRAIN_CLEAN.relative_to(PROJECT_ROOT)}")
print(f"  rows={len(train_clean)}  patients={train_clean['PatientId'].nunique()}")
print(f"Saved: {VALID_CLEAN.relative_to(PROJECT_ROOT)}")
print(f"  rows={len(valid_clean)}  patients={valid_clean['PatientId'].nunique()}")

Saved: data\processed\xray\train_clean.csv
  rows=948  patients=744
Saved: data\processed\xray\valid_clean.csv
  rows=252  patients=186


## 5. Verification

Checks:
1. Zero `PatientId` overlap between `train_clean` and `valid_clean`
2. Zero `PatientId` overlap between either clean split and official `test.csv`
3. Every cleaned row still points to an existing image
4. Class-positive counts and prevalence (%) for train_clean, valid_clean, and official test

In [9]:
train_patients = set(train_clean["PatientId"])
valid_patients = set(valid_clean["PatientId"])
test_patients = set(test_raw["PatientId"])

overlap_tv = train_patients & valid_patients
overlap_train_test = train_patients & test_patients
overlap_valid_test = valid_patients & test_patients

print(f"PatientId overlap train_clean ∩ valid_clean : {len(overlap_tv)}")
print(f"PatientId overlap train_clean ∩ test        : {len(overlap_train_test)}")
print(f"PatientId overlap valid_clean ∩ test        : {len(overlap_valid_test)}")

assert len(overlap_tv) == 0, "Leakage between train_clean and valid_clean"
assert len(overlap_train_test) == 0, "Leakage between train_clean and official test"
assert len(overlap_valid_test) == 0, "Leakage between valid_clean and official test"
print("\n[ok] Patient-level leakage checks passed (all overlaps = 0)")

assert_images_exist(train_clean, "train_clean.csv")
assert_images_exist(valid_clean, "valid_clean.csv")

PatientId overlap train_clean ∩ valid_clean : 0
PatientId overlap train_clean ∩ test        : 0
PatientId overlap valid_clean ∩ test        : 0

[ok] Patient-level leakage checks passed (all overlaps = 0)
[ok] train_clean.csv: all 948 image paths exist
[ok] valid_clean.csv: all 252 image paths exist


In [10]:
def prevalence_table(df: pd.DataFrame, name: str) -> pd.DataFrame:
    positives = df[LABEL_COLS].sum().astype(int)
    prevalence = (100.0 * positives / len(df)).round(2)
    table = pd.DataFrame({
        "positives": positives,
        "prevalence_%": prevalence,
    })
    table.index.name = "disease"
    print(f"\n{name}: n={len(df)} images, patients={df['PatientId'].nunique()}")
    display(table)
    return table


train_prev = prevalence_table(train_clean, "train_clean")
valid_prev = prevalence_table(valid_clean, "valid_clean")
test_prev = prevalence_table(test_raw, "official test (unchanged)")

print("\nHernia positives (rare class reminder):")
print(f"  train_clean : {int(train_clean['Hernia'].sum())}")
print(f"  valid_clean : {int(valid_clean['Hernia'].sum())}")
print(f"  official test: {int(test_raw['Hernia'].sum())}")


train_clean: n=948 images, patients=744


,positives,prevalence_%
disease,,
Atelectasis,102,10.76
Cardiomegaly,17,1.79
Consolidation,36,3.80
Edema,10,1.05
Effusion,132,13.92
Emphysema,15,1.58
Fibrosis,8,0.84
Hernia,2,0.21
Infiltration,165,17.41



valid_clean: n=252 images, patients=186


,positives,prevalence_%
disease,,
Atelectasis,23,9.13
Cardiomegaly,7,2.78
Consolidation,4,1.59
Edema,8,3.17
Effusion,23,9.13
Emphysema,1,0.40
Fibrosis,7,2.78
Hernia,1,0.40
Infiltration,39,15.48



official test (unchanged): n=420 images, patients=389


,positives,prevalence_%
disease,,
Atelectasis,60,14.29
Cardiomegaly,50,11.90
Consolidation,53,12.62
Edema,50,11.90
Effusion,53,12.62
Emphysema,56,13.33
Fibrosis,61,14.52
Hernia,50,11.90
Infiltration,59,14.05



Hernia positives (rare class reminder):
  train_clean : 2
  valid_clean : 1
  official test: 50


## 6. Conclusion

- **Raw course data remains unchanged** under `data/raw/xray/nih/` and `data_dl/`.
- The new validation split is **leakage-safe** at the patient level
  (`train_clean` ∩ `valid_clean` = 0 shared `PatientId`s).
- Rare classes, **especially Hernia**, remain highly imbalanced.
- **Class weights must be calculated later from `train_clean` only**
  (never from validation or test).
- The **official test set remains untouched** until final evaluation.